# Deep Reinforcement Learning (DRL) - Exemplo de Uso

Este notebook demonstra como usar o agente DRL treinado para fazer inferências.

## Índice
1. Configuração e Imports
2. Teste do Ambiente de Treino
3. Inferência com Modelo Treinado
4. Comparação com Estratégias Tradicionais

## 1. Configuração e Imports

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
from pathlib import Path

# Adiciona raiz do projeto ao path
project_root = Path.cwd().parent.parent if 'notebooks' in str(Path.cwd()) else Path.cwd()
sys.path.insert(0, str(project_root))

# Imports do projeto
from src.environments.trading_env import TradingEnv
from src.agents.drl_agent import DDQNAgent
from src.strategies.drl_strategy import DRLStrategy
from src.data_handler.provider import get_provider_instance
from src.simulation.engine import SimulationEngine

import yaml

print(f"Projeto root: {project_root}")

## 2. Teste do Ambiente de Treino

Vamos testar se o ambiente de trading está funcionando corretamente.

In [ ]:
# Carrega configuração
with open(project_root / 'configs' / 'main.yaml', 'r', encoding='utf-8') as f:
    config = yaml.safe_load(f)

# Encontra config do ativo WDO$ e sua estratégia DRL
asset_config = None
drl_strategy_config = None

for asset in config.get('assets', []):
    if asset.get('ticker') == 'WDO$':
        asset_config = asset
        # Busca estratégia DRL dentro do asset
        for strategy in asset.get('strategies', []):
            if strategy.get('name') == 'DRLStrategy':
                drl_strategy_config = strategy
                break
        break

if asset_config is None:
    raise ValueError("Ativo WDO$ não encontrado no config")

if drl_strategy_config is None:
    raise ValueError("Estratégia DRLStrategy não encontrada para WDO$")

print("Configuração do ativo DRL:")
print(f"  Ticker: {asset_config['ticker']}")
print(f"  Estratégia: {drl_strategy_config['name']}")
print(f"  Provider: {drl_strategy_config['provider']}")
print(f"  Dados: {drl_strategy_config['data']['start_date']} a {drl_strategy_config['data']['end_date']}")

In [ ]:
# Instancia provider e ambiente
provider = get_provider_instance(drl_strategy_config['provider'])
env = TradingEnv(
    ticker=asset_config['ticker'],
    strategy_config=drl_strategy_config,
    provider=provider
)

print(f"\nAmbiente criado:")
print(f"  Steps disponíveis: {len(env.market_features_df)}")
print(f"  State dimension: {env.state_dim}")
print(f"  Actions: {env.action_space}")
print(f"  Market features: {env.get_feature_names()}")

In [ ]:
# Testa um episódio completo no ambiente
state = env.reset()
episode_rewards = []
episode_actions = []
episode_prices = []

done = False
step_count = 0

print("\nExecutando episódio de teste (primeiros 50 steps)...")

while not done and step_count < 50:
    # Ação aleatória para teste
    action = np.random.choice(env.action_space)
    
    # Executa step
    next_state, reward, done = env.step(action)
    
    episode_rewards.append(reward)
    episode_actions.append(action)
    episode_prices.append(env.prices[env.current_step - 1])
    
    state = next_state if not done else state
    step_count += 1

print(f"\nEpisódio completo!")
print(f"  Steps: {step_count}")
print(f"  Recompensa total: {sum(episode_rewards):.4f}")
print(f"  Recompensa média: {np.mean(episode_rewards):.4f}")
print(f"  Portfolio value final: {env.portfolio_value:.4f}")

In [ ]:
# Visualiza resultados do teste
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

# Preços
axes[0].plot(episode_prices, label='Preço', color='blue', alpha=0.7)
axes[0].set_ylabel('Preço')
axes[0].set_title('Teste do Ambiente DRL - Episódio Aleatório')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Ações
action_names = {0: 'VENDA', 1: 'HOLD', 2: 'COMPRA'}
action_colors = {0: 'red', 1: 'gray', 2: 'green'}
for i, action in enumerate(episode_actions):
    axes[1].axvline(i, color=action_colors[action], alpha=0.5, linewidth=2)
axes[1].set_ylabel('Ações')
axes[1].set_ylim(-0.5, 2.5)
axes[1].set_yticks([0, 1, 2])
axes[1].set_yticklabels(['VENDA', 'HOLD', 'COMPRA'])
axes[1].grid(True, alpha=0.3)

# Recompensas acumuladas
cumulative_rewards = np.cumsum(episode_rewards)
axes[2].plot(cumulative_rewards, label='Recompensa Acumulada', color='purple', linewidth=2)
axes[2].axhline(0, color='black', linestyle='--', linewidth=1, alpha=0.5)
axes[2].set_ylabel('Recompensa Acumulada')
axes[2].set_xlabel('Step')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Inferência com Modelo Treinado

Agora vamos testar o modelo DRL treinado (se disponível).

In [ ]:
# Verifica se modelo treinado existe
models_dir = Path(config.get('global_settings', {}).get('model_directory', 'models'))
model_path = models_dir / f"{asset_config['ticker']}_DRLStrategy_prod_drl.keras"

if model_path.exists():
    print(f"✓ Modelo encontrado: {model_path}")
    
    # Carrega modelo usando DRLStrategy
    strategy = DRLStrategy()
    model = strategy.load(str(models_dir / f"{asset_config['ticker']}_DRLStrategy_prod"))
    
    print(f"  Input shape: {model.input_shape}")
    print(f"  Output shape: {model.output_shape}")
    print(f"  Lookback: {model.lookback}")
    
else:
    print(f"✗ Modelo não encontrado: {model_path}")
    print(f"  Execute: poetry run python train_drl_model.py")
    model = None

In [ ]:
# Se modelo existe, testa inferência manual
if model is not None:
    # Reset ambiente
    state = env.reset()
    
    print("\nTestando inferência com modelo treinado...")
    print(f"Estado inicial shape: {state.shape}")
    
    # Inferência
    state_reshaped = state.reshape(1, -1)
    q_values = model.predict(state_reshaped, verbose=0)[0]
    action = int(np.argmax(q_values))
    
    action_names = {0: 'VENDA', 1: 'HOLD', 2: 'COMPRA'}
    
    print(f"\nResultado da inferência:")
    print(f"  Q-values: {q_values}")
    print(f"  Ação escolhida: {action} ({action_names[action]})")
    print(f"  Confiança: {np.max(q_values):.4f}")

In [ ]:
# Executa episódio completo com modelo treinado
if model is not None:
    state = env.reset()
    episode_rewards_trained = []
    episode_actions_trained = []
    episode_q_values = []
    
    done = False
    step_count = 0
    max_steps = 100
    
    print(f"\nExecutando episódio com agente treinado ({max_steps} steps)...")
    
    while not done and step_count < max_steps:
        # Inferência com modelo
        state_reshaped = state.reshape(1, -1)
        q_values = model.predict(state_reshaped, verbose=0)[0]
        action = int(np.argmax(q_values))
        
        # Executa step
        next_state, reward, done = env.step(action)
        
        episode_rewards_trained.append(reward)
        episode_actions_trained.append(action)
        episode_q_values.append(q_values)
        
        state = next_state if not done else state
        step_count += 1
    
    print(f"\nResultados:")
    print(f"  Steps: {step_count}")
    print(f"  Recompensa total: {sum(episode_rewards_trained):.4f}")
    print(f"  Recompensa média: {np.mean(episode_rewards_trained):.4f}")
    print(f"  Portfolio value final: {env.portfolio_value:.4f}")
    print(f"  ROI: {(env.portfolio_value - 1.0) * 100:.2f}%")

In [ ]:
# Visualiza desempenho do agente treinado
if model is not None:
    fig, axes = plt.subplots(4, 1, figsize=(14, 12), sharex=True)
    
    # Preços
    prices_trained = env.prices[:step_count]
    axes[0].plot(prices_trained, label='Preço', color='blue', alpha=0.7)
    axes[0].set_ylabel('Preço')
    axes[0].set_title('Desempenho do Agente DRL Treinado')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Ações
    action_colors = {0: 'red', 1: 'gray', 2: 'green'}
    for i, action in enumerate(episode_actions_trained):
        axes[1].axvline(i, color=action_colors[action], alpha=0.6, linewidth=2)
    axes[1].set_ylabel('Ações')
    axes[1].set_ylim(-0.5, 2.5)
    axes[1].set_yticks([0, 1, 2])
    axes[1].set_yticklabels(['VENDA', 'HOLD', 'COMPRA'])
    axes[1].grid(True, alpha=0.3)
    
    # Q-values
    q_values_array = np.array(episode_q_values)
    axes[2].plot(q_values_array[:, 0], label='Q(VENDA)', color='red', alpha=0.7)
    axes[2].plot(q_values_array[:, 1], label='Q(HOLD)', color='gray', alpha=0.7)
    axes[2].plot(q_values_array[:, 2], label='Q(COMPRA)', color='green', alpha=0.7)
    axes[2].set_ylabel('Q-values')
    axes[2].legend()
    axes[2].grid(True, alpha=0.3)
    
    # Recompensas acumuladas
    cumulative_rewards_trained = np.cumsum(episode_rewards_trained)
    axes[3].plot(cumulative_rewards_trained, label='Recompensa Acumulada', color='purple', linewidth=2)
    axes[3].axhline(0, color='black', linestyle='--', linewidth=1, alpha=0.5)
    axes[3].fill_between(range(len(cumulative_rewards_trained)), 
                          cumulative_rewards_trained, 0, 
                          alpha=0.3, color='purple')
    axes[3].set_ylabel('Recompensa Acumulada')
    axes[3].set_xlabel('Step')
    axes[3].legend()
    axes[3].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

## 4. Teste com SimulationEngine

Agora vamos testar usando a interface oficial do `SimulationEngine`.

In [ ]:
# Instancia engine
engine = SimulationEngine(config_path=str(project_root / 'configs' / 'main.yaml'))

# Executa ciclo de simulação
target_date = datetime(2025, 10, 15, 12, 0)

print(f"Executando simulação via SimulationEngine...")
print(f"  Asset: WDO$")
print(f"  Strategy: DRLStrategy")
print(f"  Data: {target_date}")
print(f"  Timeframe: D1")

result = engine.run_simulation_cycle(
    asset_symbol="WDO$",
    timeframe_str="D1",
    target_datetime_local=target_date,
    strategy_name="DRLStrategy"
)

print(f"\nResultado:")
for key, value in result.items():
    if key not in ['indicators', 'raw_data']:  # Ignora dados muito grandes
        print(f"  {key}: {value}")

In [ ]:
# Executa múltiplos ciclos para análise
print("\nExecutando múltiplos ciclos de simulação...")

test_dates = pd.date_range(start='2025-10-01', end='2025-10-31', freq='D')
results_list = []

for date in test_dates[:20]:  # Limita a 20 para não demorar
    result = engine.run_simulation_cycle(
        asset_symbol="WDO$",
        timeframe_str="D1",
        target_datetime_local=date,
        strategy_name="DRLStrategy"
    )
    
    if 'error' not in result:
        results_list.append({
            'date': date,
            'ai_signal': result.get('ai_signal'),
            'ai_signal_code': result.get('ai_signal_code'),
            'final_decision': result.get('final_decision'),
            'current_price': result.get('current_price'),
        })

results_df = pd.DataFrame(results_list)
print(f"\nResultados coletados: {len(results_df)} ciclos")
print(results_df.head(10))

In [ ]:
# Visualiza distribuição de sinais
if len(results_df) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Distribuição de sinais
    signal_counts = results_df['ai_signal'].value_counts()
    axes[0].bar(signal_counts.index, signal_counts.values, color=['red', 'green', 'gray'])
    axes[0].set_xlabel('Sinal')
    axes[0].set_ylabel('Frequência')
    axes[0].set_title('Distribuição de Sinais DRL')
    axes[0].grid(True, alpha=0.3, axis='y')
    
    # Timeline de sinais
    signal_map = {'VENDA': 0, 'HOLD': 1, 'COMPRA': 2}
    results_df['signal_code'] = results_df['ai_signal'].map(signal_map)
    
    axes[1].plot(results_df['date'], results_df['current_price'], 
                 label='Preço', color='blue', alpha=0.5)
    
    # Marca sinais no gráfico
    for signal, color in [('VENDA', 'red'), ('HOLD', 'gray'), ('COMPRA', 'green')]:
        mask = results_df['ai_signal'] == signal
        axes[1].scatter(results_df[mask]['date'], 
                        results_df[mask]['current_price'],
                        color=color, label=signal, s=100, alpha=0.7, marker='o')
    
    axes[1].set_xlabel('Data')
    axes[1].set_ylabel('Preço')
    axes[1].set_title('Sinais DRL ao Longo do Tempo')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    axes[1].tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    plt.show()

## Conclusões

Este notebook demonstrou:

1. ✅ **Ambiente de Trading**: Funciona corretamente, calculando recompensas baseadas em log returns
2. ✅ **Modelo DRL**: Carrega e faz inferências (se treinado)
3. ✅ **SimulationEngine**: Integração completa com a arquitetura existente

### Próximos Passos

- Treinar o modelo com mais episódios
- Comparar performance DRL vs LSTM/RF
- Testar em diferentes condições de mercado
- Ajustar hiperparâmetros baseado em resultados

Para treinar o modelo, execute:
```bash
poetry run python train_drl_model.py
```